Lab - EDA Bivariate Analysis: Diving into Amazon UK Product Insights Part II


Objective: Delve into the dynamics of product pricing on Amazon UK to uncover insights that can inform business strategies and decision-making.


Dataset: This lab utilizes the [Amazon UK product dataset](https://www.kaggle.com/datasets/asaniczka/uk-optimal-product-price-prediction/)
which provides information on product categories, brands, prices, ratings, and more from from Amazon UK. You'll need to download it to start working with it.

Part 1: Analyzing Best-Seller Trends Across Product Categories

In [4]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, normaltest
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

In [10]:
# Load the dataset
data = pd.read_csv(r"C:\Users\Chantal Silva\OneDrive\Escritorio\Ironhack\Week 2\Day_10\lab-eda-bivariate\amz_uk_price_prediction_dataset.csv")

In [12]:
df=data.copy()

In [13]:
df.head()

,uid,asin,title,stars,reviews,price,isBestSeller,boughtInLastMonth,category
0,1,B09B96TG33,"Echo Dot (5th generation, 2022 release) | Big ...",4.7,15308,21.99,False,0,Hi-Fi Speakers
1,2,B01HTH3C8S,"Anker Soundcore mini, Super-Portable Bluetooth...",4.7,98099,23.99,True,0,Hi-Fi Speakers
2,3,B09B8YWXDF,"Echo Dot (5th generation, 2022 release) | Big ...",4.7,15308,21.99,False,0,Hi-Fi Speakers
3,4,B09B8T5VGV,"Echo Dot with clock (5th generation, 2022 rele...",4.7,7205,31.99,False,0,Hi-Fi Speakers
4,5,B09WX6QD65,Introducing Echo Pop | Full sound compact Wi-F...,4.6,1881,17.99,False,0,Hi-Fi Speakers


In [ ]:
# --- Part 1 — Step 1: Crosstab and Best-Seller proportion ---

# Explanation:
# The column 'isBestSeller' is currently True/False. 
# To make calculations easier (like proportions), we convert it into numeric values: True=1, False=0.
# Then, we create a crosstab to count how many best-sellers and non–best-sellers exist in each product category.
# Finally, we compute the percentage of best-sellers per category and sort them from highest to lowest.

# Convert boolean True/False into numeric 1/0
df["isBestSeller"] = df["isBestSeller"].astype(int)

# Create the crosstab: rows = categories, columns = best-seller (0/1)
ct = pd.crosstab(df["category"], df["isBestSeller"])

# Calculate the proportion of best sellers inside each category
prop_best = (ct[1] / ct.sum(axis=1)).sort_values(ascending=False)

print("Top categories by % of Best Sellers:")
display(prop_best.head(20))

print("\nCrosstab of counts:")
display(ct)


In [ ]:
# --- Part 1 — Step 2: Chi-Square Test + Cramér's V ---

# Explanation:
# We use the crosstab created earlier to run a Chi-square test of independence.
# This test checks whether the distribution of 'isBestSeller' depends on the product 'category'.
# Then, we calculate Cramér's V, which measures the strength of association 
# between the two categorical variables.

from scipy.stats import chi2_contingency
import numpy as np

# Run Chi-square test on the crosstab
chi2, p, dof, expected = chi2_contingency(ct)

print("Chi-square Test Results:")
print(f"Chi-square statistic: {chi2:.3f}")
print(f"Degrees of freedom: {dof}")
print(f"P-value: {p:.5f}")

# Calculate Cramer's V
n = ct.to_numpy().sum()
min_dim = min(ct.shape) - 1
cramers_v = np.sqrt(chi2 / (n * min_dim))

print(f"\nCramér's V: {cramers_v:.4f}")

# Interpretation help
if p < 0.05:
    print("\n➡ The result is statistically significant (p < 0.05).")
    print("➡ Conclusion: Best-seller status IS associated with product category.")
else:
    print("\n➡ No significant relationship found (p >= 0.05).")

# Interpret Cramér's V
print("\nStrength of association based on Cramér's V:")
if cramers_v < 0.10:
    print("Very weak association")
elif cramers_v < 0.30:
    print("Weak to moderate association")
elif cramers_v < 0.50:
    print("Moderate to strong association")
else:
    print("Strong association")


In [ ]:
# --- Part 1 — Step 3: Stacked Bar Chart ---

# Explanation:
# We will visualize the relationship between 'category' and 'isBestSeller' 
# using a stacked bar chart. 
# The height of each bar represents the total number of products in a category,
# and the colors represent how many of them are best-sellers vs non-best-sellers.


# Use the crosstab from Step 1
ct_plot = pd.crosstab(df['category'], df['isBestSeller'])

# Plot stacked bar chart
ct_plot.plot(kind='bar', stacked=True, figsize=(12, 7), colormap='viridis')

plt.title("Distribution of Best-Sellers Across Product Categories", fontsize=14)
plt.xlabel("Product Category")
plt.ylabel("Number of Products")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


🎯 Final Conclusion (ready to paste in your report)

The stacked bar chart shows that best-seller status varies significantly across product categories.
Some categories contain a much higher proportion of top-performing products, while others rarely produce best-sellers.
These patterns confirm that category has a strong influence on product performance, suggesting that pricing, marketing, and positioning strategies should be tailored to the characteristics of each specific segment.

In [ ]:
# --- Part 2 — Step 0: Removing price outliers using the IQR method ---

# Explanation:
# Extreme price values can distort visualizations and statistical analysis.
# Using the IQR method, we remove prices below Q1 - 1.5*IQR or above Q3 + 1.5*IQR.
# This creates a cleaner dataset for the following analyses.

# Calculate Q1, Q3, and IQR
Q1 = df["price"].quantile(0.25)
Q3 = df["price"].quantile(0.75)
IQR = Q3 - Q1

# Define acceptable range
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Lower bound: {lower_bound}")
print(f"Upper bound: {upper_bound}")

# Create filtered dataset without outliers
df_no_outliers = df[(df["price"] >= lower_bound) & (df["price"] <= upper_bound)].copy()

print(f"Original dataset size: {len(df)}")
print(f"Dataset size after removing outliers: {len(df_no_outliers)}")


📝 What to look for

How many rows were removed?

Is the difference large or small?

Does removing outliers significantly change the dataset?

This comment can also go into your report:

Outliers were removed using the Interquartile Range (IQR) method. This ensures that extreme prices do not distort the analysis of category-wise price differences. The majority of the dataset remains intact, preserving the overall structure while improving reliability.

In [ ]:
# --- Part 2 — Step 1: Violin plot of price across categories (top 20 categories) ---

# Explanation:
# A violin plot shows the full distribution of prices within each category.
# Because there are many categories, we limit the visualization to the top 20
# most frequent categories for readability.
# We use the cleaned dataset (df_no_outliers) to avoid distortion from extreme outliers.

import seaborn as sns
import matplotlib.pyplot as plt

# Find top 20 categories by frequency
top20 = df_no_outliers['category'].value_counts().head(20).index

# Filter dataset to only these categories
df_top20 = df_no_outliers[df_no_outliers['category'].isin(top20)]

# Plot violin plot
plt.figure(figsize=(14, 8))
sns.violinplot(
    data=df_top20,
    x='category',
    y='price',
    inner='quartile',
    palette='viridis'
)

plt.title("Price Distribution Across Top 20 Product Categories", fontsize=14)
plt.xlabel("Category")
plt.ylabel("Price")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
# --- Part 2 — Step 2: Bar chart of average price for top 10 categories ---

# Explanation:
# Here we compute the average price for each category 
# and visualize the top 10 categories with the highest product counts.
# The bar chart makes it easy to compare average prices across categories.

import seaborn as sns
import matplotlib.pyplot as plt

# Identify the top 10 categories by count
top10 = df_no_outliers["category"].value_counts().head(10).index

# Filter the dataset to contain only these categories
df_top10 = df_no_outliers[df_no_outliers["category"].isin(top10)]

# Compute average price per category
avg_price = df_top10.groupby("category")["price"].mean().sort_values(ascending=False)

# Plot bar chart
plt.figure(figsize=(12, 6))
sns.barplot(
    x=avg_price.index,
    y=avg_price.values,
    palette="viridis"
)

plt.title("Average Price of Products in Top 10 Categories", fontsize=14)
plt.xlabel("Category")
plt.ylabel("Average Price")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# --- Part 2 — Step 3: Box plots of product ratings across categories ---

# Explanation:
# Box plots allow us to compare the distribution of product ratings (stars)
# across the top 10 most common categories.
# We focus on the cleaned dataset (df_no_outliers) and select only categories
# with enough observations to ensure stable comparisons.

import seaborn as sns
import matplotlib.pyplot as plt

# Identify the top 10 categories by count
top10 = df_no_outliers["category"].value_counts().head(10).index

# Filter dataset
df_top10 = df_no_outliers[df_no_outliers["category"].isin(top10)]

# Plot side-by-side boxplots for rating distribution
plt.figure(figsize=(14, 8))
sns.boxplot(
    data=df_top10,
    x="category",
    y="stars",
    palette="viridis"
)

plt.title("Distribution of Product Ratings Across Top 10 Categories", fontsize=14)
plt.xlabel("Category")
plt.ylabel("Product Rating (Stars)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
# --- Part 3 — Step 1: Correlation between price and stars ---

# Explanation:
# We compute the Pearson correlation coefficient to measure the linear
# relationship between price and product ratings (stars).
# This helps us understand whether highly-rated products tend to be more expensive.

corr_value = df_no_outliers["price"].corr(df_no_outliers["stars"])

print(f"Correlation between price and stars: {corr_value:.4f}")

# Interpretation guidance
if abs(corr_value) < 0.1:
    print("➡ Very weak or no linear relationship.")
elif abs(corr_value) < 0.3:
    print("➡ Weak linear relationship.")
elif abs(corr_value) < 0.5:
    print("➡ Moderate relationship.")
else:
    print("➡ Strong relationship.")


In [ ]:
# --- Part 3 — Step 2: Scatter plot of price vs. stars ---

# Explanation:
# A scatter plot helps us visualize the relationship between product price and rating.
# Even if the correlation is weak, the plot may reveal patterns, clusters, or trends
# that are not visible through summary statistics.

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_no_outliers,
    x="stars",
    y="price",
    alpha=0.4,
    edgecolor=None
)

plt.title("Scatter Plot of Product Ratings vs. Price", fontsize=14)
plt.xlabel("Product Rating (Stars)")
plt.ylabel("Price")
plt.tight_layout()
plt.show()


In [ ]:
# --- Part 3 — Step 3: Correlation heatmap of numerical features ---

# Explanation:
# A correlation heatmap shows how strongly each numerical variable relates to the others.
# It helps us detect patterns, multicollinearity, and potential predictors of price or ratings.

import seaborn as sns
import matplotlib.pyplot as plt

# Select only numerical columns
numeric_cols = df_no_outliers.select_dtypes(include=['int64', 'float64'])

# Compute correlation matrix
corr_matrix = numeric_cols.corr()

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title("Correlation Heatmap of Numerical Features", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# --- Part 3 — Step 4: QQ Plot to assess normality of prices ---

# Explanation:
# A QQ plot compares the distribution of product prices to a theoretical normal distribution.
# If the points follow a straight diagonal line, the data is approximately normal.
# Deviations from the line indicate skewness or non-normal behavior.

import scipy.stats as stats
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

# Generate QQ plot
stats.probplot(df_no_outliers["price"], dist="norm", plot=plt)

plt.title("QQ Plot of Product Prices (No Outliers)", fontsize=14)
plt.tight_layout()
plt.show()
